# Project 57 — Local Expense Processing Agent

**Parse receipt text, categorize expenses, and generate summaries — locally with Ollama.**

| Component | Choice |
|-----------|--------|
| LLM runtime | Ollama (qwen3:8b) |
| Framework | LangGraph |
| Data extraction | Structured output parsing |
| Interface | Jupyter notebook |

> **Note:** This project uses simulated receipt text. In production, you would add
> OCR (e.g., PaddleOCR or Tesseract) as a preprocessing step.

## 1 · What You Will Learn

1. How to **extract structured data** from unstructured receipt text using LLMs
2. How to build an **expense categorization** pipeline
3. How to use **LangGraph** for multi-step document processing
4. How to generate **expense reports** with category breakdowns
5. How to handle **ambiguous or incomplete** receipt data gracefully

## 2 · Why This Project Matters

Expense management is tedious. Employees take photos of receipts that need to be
parsed, categorized, and reported. An LLM-based agent can automate most of this
workflow locally, keeping sensitive financial data private.

## 3 · Environment Setup

**Prerequisites:**
- Ollama running with `qwen3:8b`

In [ ]:
# !pip install -q langchain langchain-ollama langgraph

## 4 · Imports and Configuration

In [ ]:
import json
import os
from typing import TypedDict
from datetime import datetime

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = 'qwen3:8b'
llm = ChatOllama(model=MODEL, temperature=0.1)
print(f'LLM ready: {MODEL}')

## 5 · Sample Receipt Data

We simulate OCR output — the kind of text you'd get from scanning receipts.
Some are clean, some are messy (like real OCR).

In [ ]:
RECEIPTS = [
    {
        'id': 'REC-001',
        'text': (
            'UBER TECHNOLOGIES\n'
            'Trip: Downtown to Airport\n'
            'Date: 2024-03-15\n'
            'Distance: 22.4 mi\n'
            'Fare: $34.50\n'
            'Tip: $7.00\n'
            'Total: $41.50\n'
            'Payment: Visa *4521'
        ),
    },
    {
        'id': 'REC-002',
        'text': (
            'MARRIOTT HOTEL\n'
            'Guest: John Smith\n'
            'Check-in: Mar 15, 2024\n'
            'Check-out: Mar 17, 2024\n'
            'Room: King Suite - $189/night\n'
            'Room Total: $378.00\n'
            'Tax: $45.36\n'
            'Parking: $30.00\n'
            'Total: $453.36'
        ),
    },
    {
        'id': 'REC-003',
        'text': (
            'BEST BUY #1042\n'
            '1x USB-C Hub         $29.99\n'
            '1x Wireless Mouse    $24.99\n'
            '1x HDMI Cable        $12.99\n'
            'Subtotal: $67.97\n'
            'Tax: $5.44\n'
            'TOTAL: $73.41\n'
            'Date: 03/20/2024'
        ),
    },
    {
        'id': 'REC-004',
        'text': (
            'Olive Garden\n'
            '2x Pasta Primavera   $15.99 ea\n'
            '1x Chicken Alfredo   $17.99\n'
            '3x Drinks            $4.99 ea\n'
            'Subtotal: $64.94\n'
            'Tip: $13.00\n'
            'Total: $77.94\n'
            'Date: 2024-03-18\n'
            'Party size: 3'
        ),
    },
    {
        'id': 'REC-005',
        'text': (
            'SHELL GAS STATION\n'
            'Regular Unleaded\n'
            '12.45 GAL @ $3.29/GAL\n'
            'TOTAL: $40.96\n'
            '03/19/2024 14:32\n'
            'Pump #7'
        ),
    },
]

print(f'Loaded {len(RECEIPTS)} receipts')
for r in RECEIPTS:
    print(f'  {r["id"]}: {r["text"].split(chr(10))[0]}')

## 6 · Build the Extraction Pipeline

The LLM extracts structured fields from messy receipt text.

In [ ]:
EXPENSE_CATEGORIES = [
    'Transportation', 'Lodging', 'Meals & Entertainment',
    'Office Supplies', 'Fuel', 'Software', 'Other'
]


def extract_receipt(receipt_text: str) -> dict:
    """Extract structured expense data from receipt text."""
    prompt = ChatPromptTemplate.from_messages([
        ('system',
         'You are an expense processing assistant. Extract structured data from '
         'receipt text. Respond with ONLY a JSON object (no markdown):\n'
         '{{"vendor": "...", "date": "YYYY-MM-DD", "total": 0.00, '
         '"category": "...", "items": ["..."], "tax": 0.00, '
         '"tip": 0.00, "payment_method": "...", "notes": "..."}}\n\n'
         'Valid categories: {categories}\n'
         'If a field is not found, use null.'),
        ('human', 'Receipt text:\n{receipt}'),
    ])
    chain = prompt | llm | StrOutputParser()
    raw = chain.invoke({'receipt': receipt_text,
                        'categories': ', '.join(EXPENSE_CATEGORIES)})

    # Parse JSON
    try:
        start = raw.find('{')
        end = raw.rfind('}') + 1
        if start >= 0 and end > start:
            return json.loads(raw[start:end])
    except json.JSONDecodeError:
        pass

    return {'vendor': 'PARSE_ERROR', 'raw_response': raw[:200]}


# Test with first receipt
test = extract_receipt(RECEIPTS[0]['text'])
print(json.dumps(test, indent=2))

## 7 · Build the LangGraph Processing Workflow

```
Extract Data → Validate → Categorize → Generate Report
```

In [ ]:
class ExpenseState(TypedDict):
    receipts: list
    extracted: list
    validated: list
    report: str


def extract_node(state: ExpenseState) -> ExpenseState:
    """Extract structured data from all receipts."""
    extracted = []
    for receipt in state['receipts']:
        data = extract_receipt(receipt['text'])
        data['receipt_id'] = receipt['id']
        extracted.append(data)
        print(f'  Extracted: {receipt["id"]} -> {data.get("vendor", "?")} ${data.get("total", 0)}')
    return {**state, 'extracted': extracted}


def validate_node(state: ExpenseState) -> ExpenseState:
    """Validate extracted data for completeness."""
    validated = []
    for item in state['extracted']:
        issues = []
        if not item.get('vendor') or item.get('vendor') == 'PARSE_ERROR':
            issues.append('Missing vendor')
        if not item.get('total') or item.get('total', 0) <= 0:
            issues.append('Missing or invalid total')
        if not item.get('date'):
            issues.append('Missing date')
        if item.get('category') not in EXPENSE_CATEGORIES:
            issues.append(f'Invalid category: {item.get("category")}')

        item['validation_issues'] = issues
        item['is_valid'] = len(issues) == 0
        validated.append(item)

        status = 'VALID' if item['is_valid'] else f'ISSUES: {", ".join(issues)}'
        print(f'  {item.get("receipt_id", "?")}: {status}')

    return {**state, 'validated': validated}


def report_node(state: ExpenseState) -> ExpenseState:
    """Generate an expense summary report."""
    items = state['validated']

    # Calculate totals by category
    by_category = {}
    grand_total = 0
    for item in items:
        cat = item.get('category', 'Other')
        total = item.get('total', 0) or 0
        by_category[cat] = by_category.get(cat, 0) + total
        grand_total += total

    # Build report
    report_lines = [
        'EXPENSE REPORT',
        '=' * 50,
        f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
        f'Total Receipts: {len(items)}',
        f'Valid: {sum(1 for i in items if i.get("is_valid"))}',
        f'Issues: {sum(1 for i in items if not i.get("is_valid"))}',
        '',
        'BREAKDOWN BY CATEGORY:',
        '-' * 30,
    ]
    for cat, amount in sorted(by_category.items(), key=lambda x: -x[1]):
        pct = (amount / grand_total * 100) if grand_total > 0 else 0
        report_lines.append(f'  {cat:25s} ${amount:>10,.2f}  ({pct:.1f}%)')

    report_lines.append('-' * 30)
    report_lines.append(f'  {"GRAND TOTAL":25s} ${grand_total:>10,.2f}')
    report_lines.append('')
    report_lines.append('DETAIL:')
    for item in items:
        report_lines.append(
            f'  {item.get("receipt_id", "?"):8s} | {item.get("vendor", "?"):20s} | '
            f'${item.get("total", 0):>8.2f} | {item.get("category", "Other")}'
        )

    report = '\n'.join(report_lines)
    return {**state, 'report': report}


print('Processing functions defined.')

## 8 · Compile and Run the Workflow

In [ ]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(ExpenseState)
workflow.add_node('extract', extract_node)
workflow.add_node('validate', validate_node)
workflow.add_node('report', report_node)

workflow.set_entry_point('extract')
workflow.add_edge('extract', 'validate')
workflow.add_edge('validate', 'report')
workflow.add_edge('report', END)

app = workflow.compile()
print('Expense processing workflow compiled.')

In [ ]:
print('Processing receipts...')
print('=' * 60)

result = app.invoke({
    'receipts': RECEIPTS,
    'extracted': [],
    'validated': [],
    'report': '',
})

print('\n')
print(result['report'])

## 9 · LLM-Generated Insights

Ask the LLM to analyze the expense report and provide recommendations.

In [ ]:
insight_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a financial analyst. Given an expense report, provide:\n'
     '1. Top spending observations (3 points)\n'
     '2. Policy compliance notes\n'
     '3. Cost-saving suggestions\n'
     'Be specific with numbers.'),
    ('human', '{report}'),
])
chain = insight_prompt | llm | StrOutputParser()
insights = chain.invoke({'report': result['report']})

print('EXPENSE INSIGHTS')
print('=' * 40)
print(insights)

## 10 · Save Results

In [ ]:
output = {
    'extracted': result['validated'],
    'report': result['report'],
    'insights': insights,
}
with open('expense_results.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False, default=str)
print('Results saved to expense_results.json')

## 11 · Limitations

- **Simulated OCR** — real receipts need image-to-text preprocessing
- **JSON parsing** can fail with small models on complex receipts
- **Category accuracy** depends on the LLM's understanding of expense policies
- **No duplicate detection** across receipts
- Currency handling is basic (USD assumed)

## 12 · How to Improve

1. **Add real OCR**: integrate PaddleOCR or Tesseract for image receipts
2. **Policy engine**: add configurable expense rules (per-diem limits, etc.)
3. **Duplicate detection**: check for duplicate receipts by amount/date/vendor
4. **Multi-currency**: handle receipts in different currencies
5. **Export to accounting**: generate CSV/Excel for import into accounting software

## 13 · Mini Challenge

1. Add a receipt with intentionally messy OCR text and see how the extraction handles it
2. Add a policy violation check (e.g., meals over $100 per person)
3. Generate a pie chart of expenses by category using matplotlib

## 14 · Key Takeaways

| Concept | What You Practiced |
|---------|-------------------|
| Structured extraction | LLM extracts JSON from unstructured text |
| Multi-step pipeline | Extract → Validate → Report with LangGraph |
| Data validation | Check extracted fields for completeness |
| Automated reporting | Category breakdowns and financial summaries |
| Local-first | All processing runs locally with Ollama |